# 01 · Data cleaning

This notebook runs `src/cleaning.py`. It reads the Kaggle database, checks every table, and builds
one row per player and year in `data/cleaned_player_years.csv.gz`. Then it shows the main checks.

A row is a player who started at least one league match in the season that just ended. On 1 August
we know his latest rating. One year later we look at his latest rating again. If it went up by at
least the year's median change + 3 points, the row is a **breakout**.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT))
import pandas as pd
from src import cleaning
from src.common import CLEANED, DATA, TABLES
pd.set_option("display.width", 160, "display.max_columns", 20, "display.max_colwidth", 80)
cleaning.main()

 year  candidates  current_rating  paired_ratings  full_window last_source_rating  no_followup  no_followup_rate
 2009        3754            3662            3317         True         2016-07-07          345            0.0942
 2010        4022            3924            3535         True         2016-07-07          389            0.0991
 2011        4187            4125            3618         True         2016-07-07          507            0.1229
 2012        4275            4228            3741         True         2016-07-07          487            0.1152
 2013        4301            4287            3791         True         2016-07-07          496            0.1157
 2014        4140            4133            3629         True         2016-07-07          504            0.1219
 2015        4470            4451            3871        False         2016-07-07          580            0.1303


cohort          strategy  year  players  breakouts  watchlist    hits  precision_at_10pct  recall_at_10pct  average_precision  prevalence
   all  EA potential gap  2011     3618        729        361 160.000               0.443            0.219              0.364       0.201
   all random (expected)  2011     3618        729        361  72.739               0.201            0.100              0.201       0.201
   all  EA potential gap  2012     3741        682        374 162.000               0.433            0.238              0.335       0.182
   all random (expected)  2012     3741        682        374  68.182               0.182            0.100              0.182       0.182
   all  EA potential gap  2013     3791        430        379 137.000               0.361            0.319              0.331       0.113
   all random (expected)  2013     3791        430        379  42.989               0.113            0.100              0.113       0.113
 young  EA potential gap  2011    

## Rows per year

`paired_ratings` is the number of rows we can use: the player has a rating both before and after 1 August. 2015 is not complete because the data stops on 7 July 2016.

In [2]:
pd.read_csv(DATA / "coverage.csv")

,year,candidates,current_rating,paired_ratings,full_window,last_source_rating,no_followup,no_followup_rate
0,2009,3754,3662,3317,True,2016-07-07,345,0.0942
1,2010,4022,3924,3535,True,2016-07-07,389,0.0991
2,2011,4187,4125,3618,True,2016-07-07,507,0.1229
3,2012,4275,4228,3741,True,2016-07-07,487,0.1152
4,2013,4301,4287,3791,True,2016-07-07,496,0.1157
5,2014,4140,4133,3629,True,2016-07-07,504,0.1219
6,2015,4470,4451,3871,False,2016-07-07,580,0.1303


## Every check we ran

Each line is one check, the number we found, and what we did about it.

In [3]:
pd.read_csv(DATA / "cleaning_audit.csv")

,area,check,value,action
0,ratings,rows,183978,NaN
1,ratings,duplicate row ids,0,NaN
2,ratings,rows with no overall rating (all fields empty),836,removed
3,ratings,attacking_work_rate: values outside low/medium/high,4286,missing in features.py
4,ratings,defensive_work_rate: values outside low/medium/high,6823,missing in features.py
5,ratings,numeric fields outside 0-100,0,checked
6,ratings,valid rows missing the same 7 skills (dated 2007-02 to 2010-02),1877,kept; left missing for imputation
7,ratings,valid rows with potential < overall,1071,kept (EA's own values)
8,ratings,duplicate valid player/date records,0,keep the last by id
9,ratings,"isolated spikes (>= 8 from both neighbours, which agree within 2)",21,removed


## Who we lose

Players with no rating a year later cannot be labelled, so they drop out. They are older, rated lower and play in smaller leagues, so our results describe players who stay in EA's database.

In [4]:
print((DATA / "selection_bias.csv").read_text())

# Part 1: means and medians, excluded vs included starters
variable,excluded_mean,included_mean,excluded_median,included_median
age,27.5798,26.1398,27.0,26.0
overall_rating,64.799,70.292,64.0,71.0
potential_gap,3.5357,4.8321,3.0,4.0
starts,10.698,16.9271,8.0,17.0
start_share,0.3155,0.4732,0.25,0.4474
young,0.3017,0.3799,0.0,0.0
# Part 2: exclusion rate by year and league
variable,group,candidates,included,excluded_rate
all,all,28810,25502,0.1148
year,2009,3662,3317,0.0942
year,2010,3924,3535,0.0991
year,2011,4125,3618,0.1229
year,2012,4228,3741,0.1152
year,2013,4287,3791,0.1157
year,2014,4133,3629,0.1219
year,2015,4451,3871,0.1303
league,Belgium Jupiler League,2170,1732,0.2018
league,England Premier League,3258,3164,0.0289
league,France Ligue 1,3253,3045,0.0639
league,Germany 1. Bundesliga,2879,2755,0.0431
league,Italy Serie A,3396,3230,0.0489
league,Netherlands Eredivisie,2677,2238,0.164
league,Poland Ekstraklasa,1682,1355,0.1944
league,Portugal Liga ZON Sagres,2602,2011,0.2271
league

## Missing line-ups

The league-seasons with the most missing player IDs in the line-ups. Starts are undercounted there, so we keep a coverage column for each club.

In [5]:
lineups = pd.read_csv(DATA / "lineup_coverage.csv")
lineups.sort_values("mean_missing_ids", ascending=False).head(10)

,league_id,league,season,team_lineups,mean_missing_ids,share_lineups_fully_missing,share_slots_without_y,matches,league_match_coverage
48,15722,Poland Ekstraklasa,2008/2009,480,8.9729,0.5708,1.0000,240,1.0
0,1,Belgium Jupiler League,2008/2009,612,7.7827,0.6797,0.6797,306,1.0
40,13274,Netherlands Eredivisie,2008/2009,612,7.5425,0.6699,0.6967,306,1.0
80,24558,Switzerland Super League,2008/2009,360,6.5528,0.5778,0.5778,180,1.0
49,15722,Poland Ekstraklasa,2009/2010,480,5.6958,0.0000,1.0000,240,1.0
56,17642,Portugal Liga ZON Sagres,2008/2009,480,5.4125,0.4333,0.4411,240,1.0
50,15722,Poland Ekstraklasa,2010/2011,480,4.5833,0.0000,1.0000,240,1.0
51,15722,Poland Ekstraklasa,2011/2012,480,3.5479,0.0000,1.0000,240,1.0
52,15722,Poland Ekstraklasa,2012/2013,480,2.0750,0.0000,1.0000,240,1.0
57,17642,Portugal Liga ZON Sagres,2009/2010,480,0.4396,0.0000,0.0000,240,1.0


## One cleaned row

Mark Davies on 1 August 2011: rated 72 with potential 78, rated 75 a year later. The median change that year was 0, so +3 is a breakout.

In [6]:
cleaned = pd.read_csv(CLEANED)
cleaned[(cleaned.player_name == "Mark Davies") & (cleaned.year == 2011)][
    ["year", "player_name", "age", "starts", "overall_rating", "potential", "potential_gap",
     "next_rating", "change", "annual_median_change", "breakout"]]

,year,player_name,age,starts,overall_rating,potential,potential_gap,next_rating,change,annual_median_change,breakout
9948,2011,Mark Davies,23,9,72,78,6,75,3,0,1


## Baseline

Ranking players by EA's potential minus overall rating, against a random list of the same size (10% of each year's players).

In [7]:
pd.read_csv(TABLES / "baseline.csv").round(3)

,cohort,strategy,year,players,breakouts,watchlist,hits,precision_at_10pct,recall_at_10pct,average_precision,prevalence
0,all,EA potential gap,2011,3618,729,361,160.000,0.443,0.220,0.364,0.202
1,all,random (expected),2011,3618,729,361,72.739,0.202,0.100,0.202,0.202
2,all,EA potential gap,2012,3741,682,374,162.000,0.433,0.238,0.335,0.182
3,all,random (expected),2012,3741,682,374,68.182,0.182,0.100,0.182,0.182
4,all,EA potential gap,2013,3791,430,379,137.000,0.362,0.319,0.331,0.113
5,all,random (expected),2013,3791,430,379,42.989,0.113,0.100,0.113,0.113
6,young,EA potential gap,2011,1403,479,140,85.000,0.607,0.178,0.464,0.341
7,young,random (expected),2011,1403,479,140,47.798,0.341,0.100,0.341,0.341
8,young,EA potential gap,2012,1449,422,144,75.000,0.521,0.178,0.401,0.291
9,young,random (expected),2012,1449,422,144,41.938,0.291,0.099,0.291,0.291
